# Pairwise Contrastive Fine-Tuning for BLiMP

This notebook fine-tunes a morphology-aware GPT-2 model using minimal pairs with margin ranking loss.

**Goal:** Improve BLiMP score from 61.81% to 63-65%+

**Estimated time:** 1-2 hours on T4 GPU

## Setup

In [ ]:
# Install dependencies
!pip install -q transformers sentencepiece jsonlines torch datasets

In [ ]:
# Mount Google Drive (optional, for loading/saving models)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Create directory structure
!mkdir -p /content/BabyLM-Tiny/core
!mkdir -p /content/BabyLM-Tiny/data
!mkdir -p /content/BabyLM-Tiny/outputs
!mkdir -p /content/BabyLM-Tiny/results
!mkdir -p /content/BabyLM-Tiny/tokenizers

%cd /content/BabyLM-Tiny

## Upload Required Files

Upload these files from your local machine:
1. **Model checkpoint**: `outputs/morphology_clean_tokenizer/` (entire directory)
2. **Tokenizer**: `tokenizers/morphology_bpe/spm.model`
3. **Training scripts**:
   - `core/margin_ranking_trainer.py`
   - `core/train_pairwise_contrastive.py`
   - `core/generate_minimal_pairs.py`
4. **Optional**: Minimal pairs if already generated: `data/minimal_pairs.jsonl`

In [ ]:
# OR copy from Google Drive if you've uploaded there
!cp -r /content/drive/MyDrive/BabyLM/morphology_clean_tokenizer /content/BabyLM-Tiny/outputs/
!cp /content/drive/MyDrive/BabyLM/spm.model /content/BabyLM-Tiny/tokenizers/
!cp /content/drive/MyDrive/BabyLM/minimal_pairs.jsonl /content/BabyLM-Tiny/data/
!cp /content/drive/MyDrive/BabyLM/train_pairwise_contrastive.py /content/BabyLM-Tiny/core
!cp /content/drive/MyDrive/BabyLM/margin_ranking_trainer.py /content/BabyLM-Tiny/core

## Step 1: Generate Minimal Pairs (Optional)

In [ ]:
# If you haven't generated pairs yet, run this:
# Skip if you already uploaded minimal_pairs.jsonl

!python3 core/generate_minimal_pairs.py \
  --output_file data/minimal_pairs.jsonl \
  --num_pairs 20000 \
  --seed 42

## Step 2: Validate Pairs

In [ ]:
# Check pair quality
import jsonlines

with jsonlines.open('data/minimal_pairs.jsonl', 'r') as reader:
    pairs = list(reader)

print(f"Total pairs: {len(pairs):,}")
print(f"\nFirst 5 examples:")
for i, pair in enumerate(pairs[:5]):
    print(f"\n{i+1}. Category: {pair['category']}")
    print(f"   Correct:   {pair['correct']}")
    print(f"   Incorrect: {pair['incorrect']}")

In [ ]:
import jsonlines
import random
import os

# 1. Define paths
input_file = "data/minimal_pairs.jsonl"
train_file = "data/train_pairs.jsonl"
val_file = "data/val_pairs.jsonl"

# 2. Load the original pairs
print(f"Reading from {input_file}...")
with jsonlines.open(input_file, 'r') as reader:
    all_pairs = list(reader)

# 3. Shuffle and Split (90% Train / 10% Validation)
random.seed(42)
random.shuffle(all_pairs)

split_idx = int(len(all_pairs) * 0.9)
train_data = all_pairs[:split_idx]
val_data = all_pairs[split_idx:]

# 4. Save the new files
with jsonlines.open(train_file, 'w') as writer:
    writer.write_all(train_data)

with jsonlines.open(val_file, 'w') as writer:
    writer.write_all(val_data)

print(f"✅ Data split complete!")
print(f"   Training pairs:   {len(train_data)}")
print(f"   Validation pairs: {len(val_data)}")

In [ ]:
# ==========================================
# EVERYTHING IN ONE CELL - FINAL FIX
# ==========================================

import os
import json
import random
import logging
import shutil
from pathlib import Path
from typing import List, Dict

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import sentencepiece as spm
from transformers import (
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    set_seed,
    EarlyStoppingCallback
)

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ==========================================
# 1. CONFIGURATION (EDIT PATHS IF NEEDED)
# ==========================================
CONFIG = {
    "model_path": "outputs/morphology_clean_tokenizer",
    "tokenizer_path": "tokenizers/spm.model",
    "pairs_file": "data/minimal_pairs.jsonl",
    "output_dir": "outputs/morphology_contrastive_finetuned",
    "epochs": 2,
    "batch_size": 16,
    "learning_rate": 3e-5,
    "weight_decay": 0.01,
    "margin": 1.0,
    "lambda_margin": 0.3,
    "max_length": 256,
    "patience": 3,
    "seed": 42,
    "device": "cuda" if torch.cuda.is_available() else "cpu"
}

set_seed(CONFIG["seed"])

# ==========================================
# 2. DATA CLASSES & TOKENIZER (FIXED)
# ==========================================

class SentencePieceTokenizerWrapper:
    def __init__(self, model_path: str):
        self.model_path = str(model_path) # Saved for save_pretrained
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(self.model_path)
        self.pad_token_id = self.sp.pad_id()
        self.eos_token_id = self.sp.eos_id()
        self.bos_token_id = self.sp.bos_id()
        self.unk_token_id = self.sp.unk_id()
        self.vocab_size = self.sp.vocab_size()

    def encode(self, text: str) -> List[int]:
        ids = self.sp.encode(text, out_type=int)
        return [self.bos_token_id] + ids + [self.eos_token_id]

    def decode(self, ids: List[int]) -> str:
        return self.sp.decode(ids)

    def __call__(self, text, truncation=True, max_length=256, padding='max_length', return_tensors='pt'):
        if isinstance(text, str): text = [text]
        encoded = []
        for t in text:
            ids = self.encode(t)
            if truncation: ids = ids[:max_length]
            if padding == 'max_length':
                ids = ids + [self.pad_token_id] * (max_length - len(ids))
            encoded.append(ids)

        if return_tensors == 'pt':
            input_ids = torch.tensor(encoded, dtype=torch.long)
            attention_mask = (input_ids != self.pad_token_id).long()
            return {'input_ids': input_ids, 'attention_mask': attention_mask}
        return {'input_ids': encoded}

    # --- ADDED THIS METHOD TO FIX THE CRASH ---
    def save_pretrained(self, save_directory, **kwargs):
        """Satisfy Trainer's requirement to save the tokenizer."""
        output_dir = Path(save_directory)
        output_dir.mkdir(parents=True, exist_ok=True)
        dest_path = output_dir / "spm.model"

        if os.path.exists(self.model_path):
            shutil.copy(self.model_path, dest_path)
            return [str(dest_path)]
        return []

class PairedDataset(Dataset):
    def __init__(self, synthetic_pairs, tokenizer, max_length=256):
        self.pairs = synthetic_pairs
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]
        good = self.tokenizer(pair['good_sentence'], truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')
        bad = self.tokenizer(pair['bad_sentence'], truncation=True, max_length=self.max_length, padding='max_length', return_tensors='pt')

        return {
            'good_input_ids': good['input_ids'][0],
            'good_attention_mask': good['attention_mask'][0],
            'bad_input_ids': bad['input_ids'][0],
            'bad_attention_mask': bad['attention_mask'][0],
        }

def collate_paired_batch(batch):
    return {
        'good_input_ids': torch.stack([x['good_input_ids'] for x in batch]),
        'good_attention_mask': torch.stack([x['good_attention_mask'] for x in batch]),
        'bad_input_ids': torch.stack([x['bad_input_ids'] for x in batch]),
        'bad_attention_mask': torch.stack([x['bad_attention_mask'] for x in batch]),
    }

# ==========================================
# 3. CUSTOM TRAINER
# ==========================================

class CustomMarginRankingTrainer(Trainer):
    def __init__(self, lambda_margin=0.3, margin=1.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_margin = lambda_margin
        self.margin = margin

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        if "good_input_ids" in inputs:
            # Good Sentence (Teacher Forcing)
            good_inputs = {
                "input_ids": inputs["good_input_ids"],
                "attention_mask": inputs["good_attention_mask"],
                "labels": inputs["good_input_ids"]
            }
            outputs_good = model(**good_inputs)
            loss_ce = outputs_good.loss

            # Bad Sentence (No labels, just score)
            bad_inputs = {
                "input_ids": inputs["bad_input_ids"],
                "attention_mask": inputs["bad_attention_mask"],
            }
            outputs_bad = model(**bad_inputs)

            # Calculate Scores
            score_good = self._compute_sentence_score(outputs_good.logits, inputs["good_input_ids"])
            score_bad = self._compute_sentence_score(outputs_bad.logits, inputs["bad_input_ids"])

            # Margin Loss
            loss_margin = torch.clamp(self.margin - (score_good - score_bad), min=0).mean()

            # Final Loss
            final_loss = loss_ce + (self.lambda_margin * loss_margin)

            if return_outputs:
                return final_loss, outputs_good
            return final_loss

        return super().compute_loss(model, inputs, return_outputs=return_outputs)

    def _compute_sentence_score(self, logits, input_ids):
        # Shift logits and labels
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = input_ids[..., 1:].contiguous()

        # Gather log-probs
        loss_fct = torch.nn.CrossEntropyLoss(reduction='none')
        token_losses = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        token_losses = token_losses.view(shift_labels.size())

        # Mask padding and sum
        pad_id = self.processing_class.pad_token_id
        mask = (shift_labels != pad_id).float()
        return -(token_losses * mask).sum(dim=-1)

# ==========================================
# 4. EXECUTION SCRIPT
# ==========================================

def run_training():
    print("--- 1. Loading Resources ---")
    if not os.path.exists(CONFIG['pairs_file']):
        raise FileNotFoundError(f"Missing pairs file: {CONFIG['pairs_file']}")

    tokenizer = SentencePieceTokenizerWrapper(CONFIG['tokenizer_path'])
    model = GPT2LMHeadModel.from_pretrained(CONFIG['model_path']).to(CONFIG['device'])

    print("--- 2. Preparing Data ---")
    # Load and normalize pairs
    with open(CONFIG['pairs_file'], 'r') as f:
        import jsonlines
        with jsonlines.open(CONFIG['pairs_file']) as reader:
            raw_pairs = list(reader)

    # Normalize keys
    pairs = []
    for p in raw_pairs:
        if 'correct' in p: pairs.append({'good_sentence': p['correct'], 'bad_sentence': p['incorrect']})
        elif 'good_sentence' in p: pairs.append(p)

    # Split Data (90/10)
    random.shuffle(pairs)
    split_idx = int(len(pairs) * 0.9)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    print(f"Total Pairs: {len(pairs)} | Train: {len(train_pairs)} | Val: {len(val_pairs)}")

    train_dataset = PairedDataset(train_pairs, tokenizer, CONFIG['max_length'])
    eval_dataset = PairedDataset(val_pairs, tokenizer, CONFIG['max_length'])

    print("--- 3. initializing Trainer ---")
    training_args = TrainingArguments(
        output_dir=CONFIG['output_dir'],
        num_train_epochs=CONFIG['epochs'],
        per_device_train_batch_size=CONFIG['batch_size'],
        per_device_eval_batch_size=CONFIG['batch_size'],
        learning_rate=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay'],
        logging_steps=50,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        save_total_limit=2,
        seed=CONFIG['seed'],
        fp16=(CONFIG['device'] == 'cuda'),
        report_to=[],
        remove_unused_columns=False,
        label_names=["good_input_ids"]
    )

    trainer = CustomMarginRankingTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        data_collator=collate_paired_batch,
        lambda_margin=CONFIG['lambda_margin'],
        margin=CONFIG['margin'],
        callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG['patience'])]
    )

    print("--- 4. Starting Training ---")
    trainer.train()

    print("--- 5. Saving Model ---")
    final_dir = Path(CONFIG['output_dir']) / 'final_model4'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(final_dir)

    # Copy tokenizer manually as well to be safe
    if os.path.exists(CONFIG['tokenizer_path']):
        shutil.copy(CONFIG['tokenizer_path'], final_dir / 'spm.model')

    print(f"✅ DONE! Model saved to {final_dir}")

# Run it
run_training()

In [ ]:
# ==========================================
# SAVE MODEL UNDER A NEW NAME
# ==========================================
import shutil
from pathlib import Path

# 1. Define your new output directory name
NEW_MODEL_NAME = "morphology_contrastive_v2"  # <--- Change this to whatever you want
NEW_OUTPUT_DIR = Path("outputs") / NEW_MODEL_NAME

# 2. Create directory
NEW_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 3. Save Model and Tokenizer
print(f"Saving model to: {NEW_OUTPUT_DIR}...")
model.save_pretrained(NEW_OUTPUT_DIR)
tokenizer.save_pretrained(NEW_OUTPUT_DIR)

# 4. (Optional) Zip it for easy download from Colab
shutil.make_archive(str(NEW_OUTPUT_DIR), 'zip', str(NEW_OUTPUT_DIR))
print(f"✅ Saved and zipped! You can download {NEW_OUTPUT_DIR}.zip from the files tab.")

## Step 3: Fine-Tune with Pairwise Contrastive Loss

In [ ]:
# --- Updated Configuration ---
MODEL_PATH = "outputs/morphology_clean_tokenizer"
TOKENIZER_PATH = "tokenizers/spm.model"
TRAIN_FILE = "data/train_pairs.jsonl"    # Changed from PAIRS_FILE
VAL_FILE = "data/val_pairs.jsonl"        # New validation file
OUTPUT_DIR = "outputs/morphology_contrastive_finetuned"

# Training settings
EPOCHS = 5               # Increased because we now have early stopping
BATCH_SIZE = 16          # Reduce to 8 if you get CUDA OOM
LEARNING_RATE = 3e-5
MARGIN = 1.0
LAMBDA_MARGIN = 0.3
WEIGHT_DECAY = 0.01      # New regularization
PATIENCE = 3             # Stop if no improvement for 3 checks

# --- Run Fine-Tuning ---
!python3 core/train_pairwise_contrastive.py \
  --model_path {MODEL_PATH} \
  --tokenizer_path {TOKENIZER_PATH} \
  --pairs_file {TRAIN_FILE} \
  --validation_file {VAL_FILE} \
  --output_dir {OUTPUT_DIR} \
  --epochs {EPOCHS} \
  --batch_size {BATCH_SIZE} \
  --learning_rate {LEARNING_RATE} \
  --weight_decay {WEIGHT_DECAY} \
  --margin {MARGIN} \
  --lambda_margin {LAMBDA_MARGIN} \
  --patience {PATIENCE} \
  --eval_steps 100 \
  --save_steps 100 \
  --device cuda

## Step 4: Download Fine-Tuned Model

In [ ]:
# Zip the fine-tuned model for download
!zip -r morphology_contrastive_finetuned4.zip outputs/morphology_contrastive_finetuned/final_model4

# OR copy to Google Drive
!cp morphology_contrastive_finetuned4.zip /content/drive/MyDrive/BabyLM/

print("\n✅ Model saved! Download morphology_contrastive_finetuned.zip from the files panel or find it in your Google Drive.")

## Step 5: Test Model (Optional)

In [ ]:
# Quick test: Check if model prefers correct over incorrect
import torch
import sentencepiece as spm
from transformers import GPT2LMHeadModel

# Load model
model = GPT2LMHeadModel.from_pretrained(f"{OUTPUT_DIR}/final_model")
model.eval()

# Load tokenizer
sp = spm.SentencePieceProcessor()
sp.load(f"{OUTPUT_DIR}/final_model/spm.model")

def score_sentence(text):
    ids = sp.encode(text, out_type=int)
    ids = [sp.bos_id()] + ids + [sp.eos_id()]
    input_ids = torch.tensor([ids])

    with torch.no_grad():
        outputs = model(input_ids)
        logits = outputs.logits
        log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

        # Get log prob of each target token
        shift_log_probs = log_probs[:, :-1, :]
        shift_ids = input_ids[:, 1:]

        token_log_probs = torch.gather(shift_log_probs, 2, shift_ids.unsqueeze(-1)).squeeze(-1)
        score = token_log_probs.sum().item()

    return score

# Test on example pairs
test_pairs = [
    ("The dog runs fast.", "The dog run fast."),
    ("She is running quickly.", "She is run quickly."),
    ("These books are old.", "This books are old."),
]

print("Testing model preferences:\n")
correct_count = 0
for correct, incorrect in test_pairs:
    score_c = score_sentence(correct)
    score_i = score_sentence(incorrect)

    prefers_correct = score_c > score_i
    correct_count += prefers_correct

    symbol = "✅" if prefers_correct else "❌"
    print(f"{symbol} Correct: {score_c:.2f} | Incorrect: {score_i:.2f}")
    print(f"   '{correct}'")
    print(f"   '{incorrect}'\n")

print(f"\nModel prefers correct in {correct_count}/{len(test_pairs)} cases ({100*correct_count/len(test_pairs):.0f}%)")

## Next Steps

After downloading the fine-tuned model:

1. **Evaluate on BLiMP locally:**
   ```bash
   python core/evaluate_blimp.py \
     --model_type decoder \
     --model_path outputs/morphology_contrastive_finetuned/final_model \
     --tokenizer_path outputs/morphology_contrastive_finetuned/final_model/spm.model
   ```

2. **Compare with baseline:**
   ```bash
   python core/compare_blimp_results.py \
     --baseline results/baseline_blimp.txt \
     --finetuned results/finetuned_blimp.txt
   ```

**Expected improvement:** 61.81% → 63-65% on BLiMP